# 01 · Trayectorias: tabla persona-año (2022–2025)

**Proyecto final de Machine Learning — ¿Se pensionarán los contratistas del Estado?**

En este notebook convertimos los contratos de prestación de servicios de SECOP II en una tabla
**persona-año**: una fila por contratista y por año, con cuántos meses estuvo cubierto por contratos,
cuánto le pagaban, con cuántas entidades trabajó, cuántos huecos tuvo, etc.

**Decisiones clave**
- Años de análisis: **2022–2025**. 2026 se excluye porque está incompleto (llega al 19-oct).
- 2020 y 2021 solo se usan para medir **antigüedad** (años previos con contrato), porque en esos años muchas
  entidades seguían en SECOP I y la cobertura de SECOP II está subestimada.
- Para el año *t* se usan los contratos firmados en *t−1* y en *t*, recortados al año *t*.
  Los intervalos que se solapan se unen por persona (ordenar + máximo acumulado del fin) y se suman los días.
- Limpieza: valor > 0, fecha fin ≥ fecha inicio, sin fecha de inicio nula y duración entre 0,3 y 24 meses.
- Honorario mensual de un contrato = valor / duración en meses (días / 30,4375).
- IBC del independiente = máx(40% del honorario mensual, 1 SMMLV del año).

In [1]:
# --- Configuración común: rutas (Colab o local), constantes de ley y estilo de gráficas ---
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
from cycler import cycler
warnings.filterwarnings('ignore')

EN_COLAB = 'google.colab' in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ_DRIVE = Path('/content/drive/MyDrive/Proyecto Final SECOP')
    DATA_DIR = RAIZ_DRIVE / 'data' / 'processed'
    assert DATA_DIR.exists(), (f'No encuentro {DATA_DIR}. Sube la carpeta "Proyecto Final SECOP" (con data/processed/'
                               'secop_ps_AAAA.parquet, ver README.md) a la raíz de "Mi unidad" y vuelve a ejecutar.')
    # Las salidas quedan en "Proyecto Final SECOP/proyecto_secop/" para que las rutas relativas funcionen
    CARPETA_NB = RAIZ_DRIVE / 'proyecto_secop' / 'notebooks'
    CARPETA_NB.mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_NB)
else:
    # Local: variable de entorno SECOP_DATA o, si no existe, la estructura del README relativa al notebook
    # (proyecto_secop/ junto a data/, o proyecto_secop/ dentro de la carpeta del proyecto).
    candidatos = [os.environ.get('SECOP_DATA'), '../data/processed', '../../data/processed']
    DATA_DIR = next((Path(c) for c in candidatos if c and Path(c).exists()), None)
    assert DATA_DIR is not None, ('No encuentro los parquet. Define la variable de entorno SECOP_DATA con la ruta a '
                                  'data/processed (ver README.md) o ubica proyecto_secop/ junto a la carpeta data/.')

# Salidas con rutas relativas a la carpeta notebooks/
OUT = Path('../outputs'); FIGS = OUT / 'figs'; TABLAS = OUT / 'tablas'; INTER = Path('../data_intermedia')
for _p in (FIGS, TABLAS, INTER):
    _p.mkdir(parents=True, exist_ok=True)
RESULTADOS = OUT / 'resultados.json'

# Constantes de ley (no son datos externos: salario mínimo decretado y regla del IBC del independiente)
SMMLV = {2020: 877_803, 2021: 908_526, 2022: 1_000_000, 2023: 1_160_000,
         2024: 1_300_000, 2025: 1_423_500, 2026: 1_750_905}   # 2026: Decreto 1469 de 2025 (verificado)
FRACCION_IBC = 0.40          # el IBC del independiente es el 40% del honorario mensual
SEMANAS_PENSION = 1300       # semanas exigidas para pensión de vejez (Colpensiones)
DIAS_MES = 30.4375           # 365,25 / 12, para pasar duración en días a meses
SEMILLA = 42

# --- Estilo de gráficas para diapositivas: sobrio, letra grande, sin adornos ---
COLOR = {'azul': '#2a78d6', 'naranja': '#eb6834', 'aqua': '#1baf7a', 'amarillo': '#eda100',
         'magenta': '#e87ba4', 'verde': '#008300', 'violeta': '#6250d6', 'rojo': '#e34948',
         'gris': '#9b9a95', 'gris_claro': '#d9d8d3', 'tinta': '#0b0b0b', 'tinta2': '#52514e'}
SERIE = [COLOR[c] for c in ('azul', 'naranja', 'aqua', 'amarillo', 'magenta', 'verde', 'violeta', 'rojo')]
mpl.rcParams.update({
    'figure.figsize': (11, 6), 'figure.dpi': 100, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'figure.facecolor': 'white', 'axes.facecolor': 'white', 'font.family': 'DejaVu Sans',
    'font.size': 15, 'axes.titlesize': 20, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.titlepad': 34, 'axes.labelsize': 15, 'xtick.labelsize': 13, 'ytick.labelsize': 13,
    'legend.fontsize': 13, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#9b9a95', 'axes.labelcolor': COLOR['tinta2'], 'xtick.color': COLOR['tinta2'],
    'ytick.color': COLOR['tinta2'], 'text.color': COLOR['tinta'], 'axes.grid': True, 'axes.grid.axis': 'y',
    'grid.color': '#ebeae6', 'grid.linewidth': 0.9, 'axes.axisbelow': True, 'lines.linewidth': 2.5,
    'axes.prop_cycle': cycler(color=SERIE)})

def titulo(ax, texto, sub=None):
    """Título que cuenta el hallazgo + subtítulo gris con el detalle."""
    ax.set_title(texto, loc='left')
    if sub:
        ax.text(0, 1.035, sub, transform=ax.transAxes, fontsize=13, color=COLOR['tinta2'], va='bottom')

def fuente(fig, texto='Fuente: SECOP II (datos.gov.co), contratos de prestación de servicios a personas naturales.'):
    fig.text(0.01, -0.02, texto, fontsize=10.5, color=COLOR['gris'], ha='left', va='top')

def guardar(fig, nombre):
    ruta = FIGS / f'{nombre}.png'
    fig.savefig(ruta)
    print('Figura guardada:', ruta)

def num(x, d=0):
    """Formato numérico en español: 1.234.567,8"""
    s = f'{x:,.{d}f}'
    return s.replace(',', '§').replace('.', ',').replace('§', '.')

def pesos_millones(x, pos=None):
    return f'${num(x/1e6, 1)}M'

def pct(x, d=1):
    return f'{num(100*x, d)}%'

def guardar_resultados(seccion, datos):
    """Agrega/actualiza una sección en outputs/resultados.json."""
    todo = json.loads(RESULTADOS.read_text()) if RESULTADOS.exists() else {}
    todo[seccion] = datos
    RESULTADOS.write_text(json.dumps(todo, ensure_ascii=False, indent=2, default=float))
    print(f'resultados.json -> sección "{seccion}" actualizada')

print('Colab:', EN_COLAB, '| DATA_DIR:', DATA_DIR, '| existe:', DATA_DIR.exists())

Colab: False | DATA_DIR: ../data/processed | existe: True


## 1. Carga y limpieza por archivo
Cargamos solo las columnas necesarias. El texto del objeto se convierte de inmediato en dos indicadores
(¿el objeto menciona "profesional"? ¿menciona "apoyo a la gestión"?) y se descarta para ahorrar memoria.

In [2]:
COLS = ['id_persona', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato', 'fecha_de_firma',
        'valor_del_contrato', 'dias_adicionados', 'g_nero_representante_legal', 'departamento', 'orden',
        'sector', 'nit_entidad', 'justificacion_modalidad_de', 'objeto_del_contrato']
RENOMBRE = {'fecha_de_inicio_del_contrato': 'ini', 'fecha_de_fin_del_contrato': 'fin', 'fecha_de_firma': 'firma',
            'valor_del_contrato': 'valor', 'g_nero_representante_legal': 'genero', 'nit_entidad': 'nit',
            'justificacion_modalidad_de': 'justificacion'}
ANIOS_ARCHIVO = range(2020, 2026)
REPORTE_LIMPIEZA = {}

def cargar_limpio(anio):
    df = pd.read_parquet(DATA_DIR / f'secop_ps_{anio}.parquet', columns=COLS).rename(columns=RENOMBRE)
    obj = df.pop('objeto_del_contrato').str.upper()
    df['es_profesional'] = obj.str.contains('PROFESIONAL', regex=False).fillna(False).astype('int8')
    df['es_apoyo'] = obj.str.contains('APOYO A LA GESTI', regex=False).fillna(False).astype('int8')
    del obj
    rep = {'filas_originales': len(df),
           'valor_no_positivo': int((df.valor <= 0).sum()),
           'inicio_nulo': int(df.ini.isna().sum()),
           'fin_nulo': int(df.fin.isna().sum()),
           'fin_antes_de_inicio': int((df.fin < df.ini).sum())}
    df = df[(df.valor > 0) & df.ini.notna() & df.fin.notna() & (df.fin >= df.ini)]
    df['dur_meses'] = (df.fin - df.ini).dt.days / DIAS_MES
    rep['duracion_fuera_0.3_24_meses'] = int(((df.dur_meses < 0.3) | (df.dur_meses > 24)).sum())
    df = df[(df.dur_meses >= 0.3) & (df.dur_meses <= 24)].copy()
    df['honorario'] = df.valor / df.dur_meses
    # Hay registros con días adicionados absurdos (hasta 730.485); se acotan a 730 días (2 años)
    rep['dias_adicionados_acotados_a_730'] = int((df.dias_adicionados > 730).sum())
    df['dias_adicionados'] = df.dias_adicionados.clip(0, 730)
    df['es_nacional'] = (df.orden == 'Nacional').astype('int8')
    for c in ('genero', 'departamento', 'orden', 'sector', 'justificacion', 'nit'):
        df[c] = df[c].astype('category')
    rep['filas_limpias'] = len(df)
    REPORTE_LIMPIEZA[anio] = rep
    return df

_cache = {}
def contratos(anio):
    """Devuelve los contratos limpios firmados en `anio` (con caché de los dos últimos años)."""
    if anio not in _cache:
        _cache[anio] = cargar_limpio(anio)
        for k in sorted(_cache)[:-2]:
            del _cache[k]
    return _cache[anio]

## 2. Unión de intervalos por persona
Si una persona tiene dos contratos que se solapan (por ejemplo, uno de enero a junio y otro de mayo a
diciembre), no puede contar los días dos veces. Ordenamos por fecha de inicio, calculamos el máximo
acumulado de la fecha fin y abrimos un bloque nuevo solo cuando el siguiente contrato empieza después de que
terminó todo lo anterior. Los días cubiertos son la suma de los días (inclusivos) de cada bloque.
Entre bloques quedan los **huecos** (días sin ningún contrato).

In [3]:
def fusionar_intervalos(p, s, e):
    """p: id persona, s/e: inicio y fin (ya recortados al año). Devuelve métricas de cobertura por persona."""
    assert s.notna().all() and e.notna().all(), 'Hay fechas nulas: filtrarlas antes de fusionar'
    d = pd.DataFrame({'p': p.values, 's': s.values, 'e': e.values}).sort_values(['p', 's'], kind='mergesort')
    cm = d.groupby('p', sort=False)['e'].cummax()
    prev = cm.groupby(d['p'].values, sort=False).shift()
    nuevo = prev.isna() | (d['s'] > prev + pd.Timedelta(days=1))   # contratos contiguos se unen en un bloque
    d['blk'] = nuevo.cumsum().values
    bl = d.groupby('blk', sort=False).agg(p=('p', 'first'), s=('s', 'min'), e=('e', 'max'))
    bl['dias'] = (bl.e - bl.s).dt.days + 1
    bl['hueco'] = (bl.s - bl.groupby('p', sort=False).e.shift()).dt.days - 1
    per = bl.groupby('p', sort=False).agg(dias_cubiertos=('dias', 'sum'), n_bloques=('dias', 'size'),
                                          hueco_max_dias=('hueco', 'max'), inicio_cobertura=('s', 'min'),
                                          fin_cobertura=('e', 'max'))
    per['hueco_max_dias'] = per.hueco_max_dias.fillna(0).astype(int)
    per.index.name = 'id_persona'
    return per

def dias_del_anio(t):
    return 366 if pd.Timestamp(t, 12, 31).dayofyear == 366 else 365

def contratos_que_tocan(t):
    """Contratos firmados en t-1 y t que tienen al menos un día dentro del año t, recortados a t."""
    partes = [contratos(a) for a in (t - 1, t) if a in ANIOS_ARCHIVO]
    d = pd.concat(partes, ignore_index=True)
    a, b = pd.Timestamp(t, 1, 1), pd.Timestamp(t, 12, 31)
    d = d[(d.ini <= b) & (d.fin >= a)].copy()
    d['s'] = d.ini.clip(lower=a)
    d['e'] = d.fin.clip(upper=b)
    return d

def modal(d, col, excluir=None):
    """Valor más frecuente de `col` por persona (desempate alfabético estable)."""
    x = d[['id_persona', col]].copy()
    x[col] = x[col].astype(str)
    if excluir is not None:
        x = x[~x[col].isin(excluir)]
    c = x.groupby(['id_persona', col], observed=True).size().rename('n').reset_index()
    c = c.sort_values(['id_persona', 'n', col], ascending=[True, False, True]).drop_duplicates('id_persona')
    return c.set_index('id_persona')[col]

## 3. Construcción de la tabla persona-año
Para cada año *t* calculamos las variables pedidas. Además guardamos dos variables auxiliares:
- `cubierto_al_cierre`: 1 si la persona tenía contrato vigente el 31 de diciembre de *t* (dato del propio año), y
  `dias_sin_contrato_al_cierre`: días entre el fin de su último contrato en *t* y el 31-dic.
- `nit_modal`: la entidad con la que más contratos firmó en *t* (para variables de entidad en el notebook 04).
- `dias_firmados_para_t1`: días de *t+1* ya cubiertos por contratos **firmados hasta el 31-dic de t**, usando la
  fecha fin original (fin − días adicionados, para no usar prórrogas que pudieron hacerse en *t+1*).
  Esta variable **no** entra al modelo principal; solo se usa como análisis de sensibilidad en el notebook 04.

In [4]:
def construir_anio(t, detalle=True):
    d = contratos_que_tocan(t)
    cob = fusionar_intervalos(d.id_persona, d.s, d.e)
    cob['meses_cubiertos'] = cob.dias_cubiertos * 12 / dias_del_anio(t)
    if not detalle:
        return cob[['dias_cubiertos', 'meses_cubiertos']]
    d['dias_en_t'] = (d.e - d.s).dt.days + 1
    d['valor_en_t'] = d.honorario * d.dias_en_t / DIAS_MES          # valor prorrateado al año t
    g = d.groupby('id_persona', sort=False)
    agg = g.agg(n_contratos=('ini', 'size'), n_entidades=('nit', 'nunique'),
                honorario_mensual_mediano=('honorario', 'median'), valor_total=('valor_en_t', 'sum'),
                dias_adicionados_total=('dias_adicionados', 'sum'), share_entidad_nacional=('es_nacional', 'mean'),
                share_objeto_profesional=('es_profesional', 'mean'), share_objeto_apoyo=('es_apoyo', 'mean'))
    tab = cob.join(agg)
    tab['n_huecos'] = tab.n_bloques - 1
    tab['mes_inicio_primer_contrato'] = tab.inicio_cobertura.dt.month
    tab['cubierto_al_cierre'] = (tab.fin_cobertura == pd.Timestamp(t, 12, 31)).astype('int8')
    tab['dias_sin_contrato_al_cierre'] = (pd.Timestamp(t, 12, 31) - tab.fin_cobertura).dt.days
    tab['nit_modal'] = modal(d, 'nit')
    tab['departamento_modal'] = modal(d, 'departamento')
    tab['sector_modal'] = modal(d, 'sector')
    tab['justificacion_modal'] = modal(d, 'justificacion')
    gen = modal(d, 'genero', excluir=['No Definido', 'nan'])
    tab['genero'] = gen.reindex(tab.index).fillna('No Definido')
    # Compromisos ya firmados para t+1 (solo sensibilidad)
    todos = pd.concat([contratos(a) for a in (t - 1, t) if a in ANIOS_ARCHIVO], ignore_index=True)
    fin_orig = todos.fin - pd.to_timedelta(todos.dias_adicionados, unit='D')
    a1, b1 = pd.Timestamp(t + 1, 1, 1), pd.Timestamp(t + 1, 12, 31)
    m = (todos.ini <= b1) & (fin_orig >= a1)
    if m.any():
        f1 = fusionar_intervalos(todos.id_persona[m], todos.ini[m].clip(lower=a1), fin_orig[m].clip(upper=b1))
        tab['dias_firmados_para_t1'] = f1.dias_cubiertos.reindex(tab.index).fillna(0).astype(int)
    else:
        tab['dias_firmados_para_t1'] = 0
    tab['anio'] = t
    return tab.drop(columns=['n_bloques', 'inicio_cobertura', 'fin_cobertura'])

In [5]:
import time
cobertura = {}       # meses cubiertos por persona y año (incluye 2020-2021 para antigüedad)
tablas = []
for t in range(2020, 2026):
    t0 = time.time()
    if t < 2022:
        cob = construir_anio(t, detalle=False)
    else:
        tab = construir_anio(t, detalle=True)
        cob = tab[['dias_cubiertos', 'meses_cubiertos']]
        tablas.append(tab)
    cobertura[t] = cob
    print(f'{t}: {len(cob):>9,} personas activas  ({time.time()-t0:.0f}s)')

2020:   210,496 personas activas  (2s)


2021:   325,660 personas activas  (5s)


2022:   402,755 personas activas  (11s)


2023:   437,494 personas activas  (13s)


2024:   471,549 personas activas  (14s)


2025:   513,680 personas activas  (16s)


### Reporte de limpieza (contratos por archivo de año de firma)

In [6]:
limpieza = pd.DataFrame(REPORTE_LIMPIEZA).T
limpieza.index.name = 'anio_firma'
limpieza.to_csv(TABLAS / '01_reporte_limpieza.csv')
limpieza

,filas_originales,valor_no_positivo,inicio_nulo,fin_nulo,fin_antes_de_inicio,duracion_fuera_0.3_24_meses,dias_adicionados_acotados_a_730,filas_limpias
anio_firma,,,,,,,,
2020,278277,119,3296,0,4,1569,0,273290
2021,445670,207,5644,0,6,4788,0,435027
2022,574538,294,3403,1,6,6240,0,564594
2023,676928,475,4498,1,3,7836,1,664119
2024,709141,311,4270,4,5,6025,0,698533
2025,815766,275,3915,4,219,7209,4,804152


## 4. Antigüedad, rezago y variables pensionales
- `antiguedad_anios`: número de años previos (desde 2020) en los que la persona tuvo al menos un día de contrato.
  Ojo: su máximo crece con el año (2 en 2022, 5 en 2025), por eso el modelo usa además `antiguedad_2_anios`
  (presencia en *t−1* y *t−2*, de 0 a 2), que es comparable entre años.
- `meses_cubiertos_anio_anterior`: meses cubiertos en *t−1* (0 si no tuvo contrato).
- `ibc = máx(0,4 · honorario, SMMLV)`, `cotiza_minimo = 1` si 0,4 · honorario ≤ SMMLV.
- `semanas_cotizables = días cubiertos / 7`.

In [7]:
pa = pd.concat(tablas).reset_index()
presencia = {t: set(cobertura[t].index) for t in cobertura}
pa['antiguedad_anios'] = 0
pa['antiguedad_2_anios'] = 0
for t in range(2022, 2026):
    m = pa.anio == t
    ids = pa.loc[m, 'id_persona']
    pa.loc[m, 'antiguedad_anios'] = sum(ids.isin(presencia[y]).astype(int) for y in range(2020, t))
    pa.loc[m, 'antiguedad_2_anios'] = sum(ids.isin(presencia[y]).astype(int) for y in (t - 1, t - 2))
    pa.loc[m, 'meses_cubiertos_anio_anterior'] = ids.map(cobertura[t - 1].meses_cubiertos).fillna(0).values

pa['smmlv'] = pa.anio.map(SMMLV)
pa['ibc'] = np.maximum(FRACCION_IBC * pa.honorario_mensual_mediano, pa.smmlv)
pa['cotiza_minimo'] = (FRACCION_IBC * pa.honorario_mensual_mediano <= pa.smmlv).astype('int8')
pa['semanas_cotizables'] = pa.dias_cubiertos / 7
for c in ('departamento_modal', 'sector_modal', 'justificacion_modal', 'genero'):
    pa[c] = pa[c].astype('category')
orden_cols = ['id_persona', 'anio', 'meses_cubiertos', 'dias_cubiertos', 'semanas_cotizables', 'n_contratos',
              'n_entidades', 'honorario_mensual_mediano', 'valor_total', 'dias_adicionados_total', 'n_huecos',
              'hueco_max_dias', 'mes_inicio_primer_contrato', 'cubierto_al_cierre', 'dias_sin_contrato_al_cierre',
              'nit_modal', 'share_entidad_nacional',
              'share_objeto_profesional', 'share_objeto_apoyo', 'departamento_modal', 'genero',
              'justificacion_modal', 'sector_modal', 'antiguedad_anios', 'antiguedad_2_anios',
              'meses_cubiertos_anio_anterior', 'smmlv', 'ibc', 'cotiza_minimo', 'dias_firmados_para_t1']
pa = pa[orden_cols]
pa.to_parquet(INTER / 'persona_anio_2022_2025.parquet', index=False)
# Cobertura 2020-2025 (formato largo), útil para antigüedad y para el notebook de pensión
cob_larga = pd.concat([c.assign(anio=t) for t, c in cobertura.items()]).reset_index()
cob_larga.to_parquet(INTER / 'cobertura_2020_2025.parquet', index=False)
print(pa.shape)
pa.head()

(1825478, 30)


,id_persona,anio,meses_cubiertos,dias_cubiertos,semanas_cotizables,n_contratos,n_entidades,honorario_mensual_mediano,valor_total,dias_adicionados_total,...,genero,justificacion_modal,sector_modal,antiguedad_anios,antiguedad_2_anios,meses_cubiertos_anio_anterior,smmlv,ibc,cotiza_minimo,dias_firmados_para_t1
0,00002ada9091b162,2022,10.389041,316,45.142857,2,1,1.430406e+06,1.486733e+07,0,...,Mujer,Servicios profesionales y apoyo a la gestión,Servicio Público,1,1,5.556164,1000000,1.000000e+06,1,0
1,00007c220040b556,2022,11.375342,346,49.428571,2,1,1.960345e+06,2.111358e+07,43,...,Mujer,Servicios profesionales y apoyo a la gestión,agricultura,0,0,0.000000,1000000,1.000000e+06,1,322
2,000082887acd96d6,2022,10.947945,333,47.571429,2,1,5.200738e+06,5.686149e+07,0,...,Mujer,Servicios profesionales y apoyo a la gestión,"Vivienda, Ciudad y Territorio",1,1,4.997260,1000000,2.080295e+06,0,0
3,00008d9f515157f7,2022,10.323288,314,44.857143,3,1,3.578435e+06,3.679708e+07,61,...,Hombre,Servicios profesionales y apoyo a la gestión,deportes,1,1,5.950685,1000000,1.431374e+06,0,0
4,0000d8f04d773ca3,2022,11.769863,358,51.142857,1,1,4.399370e+06,5.174454e+07,0,...,Mujer,Servicios profesionales y apoyo a la gestión,Cultura,1,1,3.484932,1000000,1.759748e+06,0,0


In [8]:
pa.describe(percentiles=[.25, .5, .75]).T.round(2)

,count,mean,std,min,25%,50%,75%,max
anio,1825478.0,2.023600e+03,1.120000e+00,2022.00,2023.00,2024.00,2025.00,2.025000e+03
meses_cubiertos,1825478.0,7.920000e+00,3.350000e+00,0.03,5.21,9.04,10.72,1.200000e+01
dias_cubiertos,1825478.0,2.411400e+02,1.019100e+02,1.00,159.00,275.00,327.00,3.660000e+02
semanas_cotizables,1825478.0,3.445000e+01,1.456000e+01,0.14,22.71,39.29,46.71,5.229000e+01
n_contratos,1825478.0,1.640000e+00,9.100000e-01,1.00,1.00,1.00,2.00,6.100000e+01
n_entidades,1825478.0,1.110000e+00,3.900000e-01,1.00,1.00,1.00,1.00,4.200000e+01
honorario_mensual_mediano,1825478.0,4.532082e+06,2.520544e+07,0.00,2547115.72,3732848.29,5127076.31,1.610225e+10
valor_total,1825478.0,1.090594e+08,6.694828e+10,0.00,16213379.75,28729894.17,46783050.42,6.472860e+13
dias_adicionados_total,1825478.0,1.387000e+01,3.487000e+01,0.00,0.00,0.00,0.00,9.820000e+02
n_huecos,1825478.0,4.200000e-01,6.500000e-01,0.00,0.00,0.00,1.00,1.000000e+01


## 5. Validación contra el hallazgo previo de 2024
El hallazgo previo (nacional, 2024) reportó: 471.547 personas activas; meses cubiertos mediana 7,9
(p25 4,1; p75 10,3); 13,2% con 11 meses o más; 36,1% con menos de 6; honorario mensual mediano $3,95M;
37,5% cotiza sobre el mínimo; mediana de 34,4 semanas al año ⇒ unos 37,8 años para 1.300 semanas.

In [9]:
p24 = pa[pa.anio == 2024]
obtenido = {
    'personas_activas': len(p24),
    'meses_mediana': p24.meses_cubiertos.median(),
    'meses_p25': p24.meses_cubiertos.quantile(.25),
    'meses_p75': p24.meses_cubiertos.quantile(.75),
    'pct_11_meses_o_mas': (p24.meses_cubiertos >= 11).mean() * 100,
    'pct_menos_6_meses': (p24.meses_cubiertos < 6).mean() * 100,
    'honorario_mediano_millones': p24.honorario_mensual_mediano.median() / 1e6,
    'pct_cotiza_minimo': p24.cotiza_minimo.mean() * 100,
    'semanas_mediana': p24.semanas_cotizables.median(),
}
obtenido['anios_para_1300'] = SEMANAS_PENSION / obtenido['semanas_mediana']
esperado = {'personas_activas': 471547, 'meses_mediana': 7.9, 'meses_p25': 4.1, 'meses_p75': 10.3,
            'pct_11_meses_o_mas': 13.2, 'pct_menos_6_meses': 36.1, 'honorario_mediano_millones': 3.95,
            'pct_cotiza_minimo': 37.5, 'semanas_mediana': 34.4, 'anios_para_1300': 37.8}
val = pd.DataFrame({'esperado': esperado, 'obtenido': obtenido})
val['diferencia'] = val.obtenido - val.esperado
val.round(3)

,esperado,obtenido,diferencia
personas_activas,471547.00,471549.000,2.000
meses_mediana,7.90,7.934,0.034
meses_p25,4.10,4.131,0.031
meses_p75,10.30,10.328,0.028
pct_11_meses_o_mas,13.20,13.197,-0.003
pct_menos_6_meses,36.10,36.050,-0.050
honorario_mediano_millones,3.95,3.952,0.002
pct_cotiza_minimo,37.50,37.408,-0.092
semanas_mediana,34.40,34.571,0.171
anios_para_1300,37.80,37.603,-0.197


**Lectura de la validación.** El hallazgo previo se reproduce. Las cifras de meses (mediana, p25, p75,
% con 11+ meses, % con menos de 6) y el honorario mediano coinciden al redondeo reportado. Hay tres diferencias
pequeñas, y ninguna cambia las conclusiones:
- **Personas:** 471.549 frente a 471.547 (2 más, el 0,0004%).
- **% que cotiza sobre el mínimo:** 37,4% frente a 37,5% (0,1 puntos). Viene de redondeos en la duración usada
  para calcular el honorario.
- **Semanas:** la mediana exacta es **34,6 semanas** (242 días / 7) y no 34,4, así que el tiempo hasta 1.300
  semanas sale en **37,6 años** y no en 37,8. El 34,4 previo parece venir de convertir la mediana *ya redondeada*
  de 7,9 meses a semanas (7,9 × 30,4375 / 7 = 34,35). En adelante usamos la cifra exacta: 34,6 semanas,
  es decir 37,6 años.

Dos detalles de método hacen cuadrar las cifras. Primero, los días de cada bloque se cuentan de forma
**inclusiva** (fin − inicio + 1). Segundo, los meses cubiertos son días × 12 / días del año. Como 2024 fue
bisiesto, un mes equivale a 30,5 días, y "11 meses o más" significa 336 días o más.

### Decisiones de definición y su sensibilidad (revisión independiente)
Mantenemos la definición del hallazgo previo para que sea comparable, pero la documentamos:

**1. Los umbrales de "11 meses" y "9 meses" se miden en días / días del año.** "11 meses o más" significa
días × 12 / días del año ≥ 11: al menos **336 días en 2024** (bisiesto) y 335 en los años de 365 días. Un contrato
del 1-feb al 31-dic (11 meses calendario: 334 o 335 días) queda por debajo del umbral. Si "11 meses" se definiera
en meses calendario (≥ días del año − 31), el porcentaje subiría:

| Año | % con 11+ (esta definición) | % con 11+ en meses calendario |
|---|---|---|
| 2022 | 25,4% | 29,3% |
| 2023 | 17,7% | 19,6% |
| 2024 | **13,2%** | **14,7%** |
| 2025 | 17,6% | 19,1% |

Con el umbral de 9 meses del notebook 04 pasa lo mismo: del 1-ene al 30-sep son 273 días, es decir 8,975 meses,
y quedan como "menos de 9". Esto afecta entre el 0,7% y el 1,0% de las personas por año. La conclusión no cambia:
en 2024, como mucho 1 de cada 7 contratistas tuvo contrato todo el año o casi todo.

**2. El IBC usa la mediana del honorario por contrato, no la suma de contratos simultáneos.** Quien tiene dos
contratos a la vez cotiza sobre la suma de ambos ingresos. Por eso, usar la mediana sobreestima un poco el
porcentaje que cotiza sobre el mínimo. Con un ingreso efectivo (valor prorrateado en *t* / meses cubiertos) la
cifra baja levemente: 31,1% → 30,5% en 2022, 36,3% → 35,3% en 2023, **37,4% → 36,5% en 2024** y 40,0% → 38,5% en
2025. Entre quienes trabajan con más de una entidad, entre el 9% y el 13% cambia de clase. Según cómo se mensualice
el honorario, la cifra de 2024 queda entre el 36,5% y el 38,7%.

**3. Otras precisiones.**
- Solo se usan los contratos firmados en *t−1* y *t*. Los firmados en *t−2* o antes que todavía tocan *t* son
  pocos (3.199 en 2024): con ellos el % con 11+ de 2024 pasaría de 13,20% a 13,54%, y la mediana no cambia.
- `n_huecos` y `hueco_max_dias` cuentan solo huecos **internos**, entre dos contratos del año. Los meses sin
  contrato al inicio o al final del año no son huecos; quedan reflejados en `mes_inicio_primer_contrato` y en
  `dias_sin_contrato_al_cierre`.
- La fecha fin registrada incluye las prórrogas. Los contratos cedidos (0,5%) quedan a nombre del contratista
  original.

In [10]:
assert abs(obtenido['personas_activas'] - 471547) < 50
assert abs(obtenido['meses_mediana'] - 7.9) < 0.05 and abs(obtenido['pct_11_meses_o_mas'] - 13.2) < 0.1
assert abs(obtenido['semanas_mediana'] - 34.4) < 0.25 and abs(obtenido['anios_para_1300'] - 37.8) < 0.25
print('Validación 2024 OK')

Validación 2024 OK


## 6. Resumen por año y chequeos de sentido común

In [11]:
resumen_anio = pa.groupby('anio').agg(
    personas=('id_persona', 'size'), meses_mediana=('meses_cubiertos', 'median'),
    pct_11_mas=('meses_cubiertos', lambda x: (x >= 11).mean() * 100),
    pct_menos_6=('meses_cubiertos', lambda x: (x < 6).mean() * 100),
    honorario_mediano=('honorario_mensual_mediano', 'median'), pct_cotiza_minimo=('cotiza_minimo', 'mean'),
    semanas_mediana=('semanas_cotizables', 'median'), contratos_mediana=('n_contratos', 'median'),
    pct_con_hueco=('n_huecos', lambda x: (x > 0).mean() * 100))
resumen_anio['pct_cotiza_minimo'] *= 100
resumen_anio['anios_para_1300'] = SEMANAS_PENSION / resumen_anio.semanas_mediana
resumen_anio.to_csv(TABLAS / '01_resumen_por_anio.csv')
resumen_anio.round(2)

,personas,meses_mediana,pct_11_mas,pct_menos_6,honorario_mediano,pct_cotiza_minimo,semanas_mediana,contratos_mediana,pct_con_hueco,anios_para_1300
anio,,,,,,,,,,
2022,402755,9.99,25.44,28.46,3352473.21,31.06,43.43,1.0,34.34,29.93
2023,437494,9.27,17.66,25.42,3556111.00,36.31,40.29,1.0,36.26,32.27
2024,471549,7.93,13.20,36.05,3951553.76,37.41,34.57,1.0,34.68,37.60
2025,513680,9.04,17.58,26.96,4059355.40,39.97,39.29,1.0,33.98,33.09


**Ojo con comparar años.** 2024 es el año con menos cobertura (mediana de 7,9 meses). Coincide con el primer año
de los nuevos alcaldes y gobernadores, que arrancan a contratar tarde. 2022 tiene la cobertura más alta (25% con
11 meses o más) porque, por la Ley de Garantías, muchas entidades firmaron en enero contratos para todo el año.
Por eso el hallazgo de 2024 es el escenario más desfavorable de los cuatro años, y en los notebooks siguientes
también reportamos el promedio 2022–2025.

In [12]:
# Chequeos: no puede haber más de 366 días, ni meses > 12, ni valores negativos
assert pa.dias_cubiertos.between(1, 366).all()
assert (pa.meses_cubiertos <= 12.0001).all()
assert (pa.honorario_mensual_mediano > 0).all()
assert not pa.duplicated(['id_persona', 'anio']).any()
print('Personas distintas 2022-2025:', pa.id_persona.nunique())
print('Personas distintas 2020-2025 (con cobertura):', cob_larga.id_persona.nunique())
print('Distribución de género (persona-año):'); print(pa.genero.value_counts(normalize=True).round(3))

Personas distintas 2022-2025: 848267


Personas distintas 2020-2025 (con cobertura): 919734
Distribución de género (persona-año):
genero
Mujer          0.435
Hombre         0.341
No Definido    0.223
Otro           0.001
Name: proportion, dtype: float64


In [13]:
guardar_resultados('01_trayectorias', {
    'validacion_2024': {k: {'esperado': esperado[k], 'obtenido': float(obtenido[k])} for k in esperado},
    'personas_por_anio': {int(k): int(v) for k, v in resumen_anio.personas.items()},
    'resumen_por_anio': {int(a): {k: float(v) for k, v in fila.items()} for a, fila in resumen_anio.iterrows()},
    'personas_distintas_2022_2025': int(pa.id_persona.nunique()),
    'filas_persona_anio': int(len(pa)),
    'limpieza_por_archivo': {int(a): {k: int(v) for k, v in r.items()} for a, r in REPORTE_LIMPIEZA.items()},
    'definiciones': {'mes': 'dias_cubiertos * 12 / dias_del_anio', 'dias': 'inclusivos (fin - inicio + 1) por bloque unido',
                     'honorario': 'valor / ((fin - inicio) en dias / 30.4375)', 'ibc': 'max(0.4 * honorario, SMMLV)'}})

resultados.json -> sección "01_trayectorias" actualizada
